# Pelajaran 13 - Memori Agen


## Pengaturan

Notebook ini menunjukkan cara membangun agen pemesanan perjalanan dengan **memori persisten** menggunakan **Microsoft Agent Framework** (MAF).

Anda akan belajar bagaimana berbagai jenis memori agen — kerja, jangka pendek, dan jangka panjang — memengaruhi cara agen menyimpan dan menggunakan informasi selama percakapan.

**Prasyarat:**
- Proyek Microsoft Foundry dengan model chat yang sudah dideploy (misalnya `gpt-4.1-mini`).
- Masuk dengan Azure CLI — jalankan `az login` di terminal Anda.
- `AZURE_AI_PROJECT_ENDPOINT` — endpoint proyek Microsoft Foundry Anda.
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — nama model yang Anda deploy.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import json
import dotenv
from typing import Annotated
from datetime import datetime

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## Jenis Memori Agen

Agen AI dapat memanfaatkan berbagai jenis memori, masing-masing dengan tujuan yang berbeda:

### Memori Kerja
Benang percakapan itu sendiri — pesan-pesan yang dipertukarkan dalam satu sesi. Agen dapat merujuk kembali ke pesan sebelumnya dalam benang yang sama untuk menjaga koherensi. Dalam MAF ini dibuat melalui **`agent.create_session()`**, yang mengembalikan `AgentSession`.

### Memori Jangka Pendek
Informasi yang bertahan selama durasi tugas atau sesi tetapi tidak disimpan secara permanen. Misalnya, agen dapat mengumpulkan fakta selama percakapan perencanaan multi-putar dan menggunakannya untuk menghasilkan jadwal akhir.

### Memori Jangka Panjang
Preferensi dan fakta yang bertahan **antar sesi**. Pengguna yang kembali tidak perlu mengulangi batasan diet atau gaya perjalanan mereka. Memori jangka panjang biasanya didukung oleh penyimpanan eksternal — basis data, file, atau indeks vektor — dan disajikan kepada agen melalui alat.


## Memori Kerja dengan Sesi

Bentuk paling sederhana dari memori adalah sesi percakapan. Ketika Anda melewatkan sesi yang sama (dibuat melalui `agent.create_session()`) ke panggilan `agent.run()` berturut-turut, agen dapat melihat seluruh riwayat percakapan tersebut dan dapat mengingat detail sebelumnya.

Mari kita buat agen perjalanan dan demonstrasikan memori kerja.


In [ ]:
agent = client.as_agent(
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

Agen dengan benar mengingat anggaran karena kedua pesan berbagi sesi yang sama. Ini adalah **memori kerja** — yang ada hanya selama masa hidup sesi.

### Apa yang terjadi dengan utas baru?

Jika kita membuat sesi **baru**, agen tidak memiliki memori dari percakapan sebelumnya:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## Pola Memori Jangka Panjang

Untuk mengingat preferensi pengguna **di seluruh sesi**, kita membutuhkan penyimpanan yang persisten yang berada di luar alur percakapan. Agen mengakses penyimpanan ini melalui **tools** — fungsi yang dapat dipanggil untuk menyimpan dan mengambil informasi.

Di bawah ini kami mengimplementasikan penyimpanan preferensi sederhana dalam memori (dalam produksi kamu akan menggunakan database atau indeks vektor sebagai dukungan) dan mengeksposnya sebagai tools yang dapat digunakan agen.

### Arsitektur
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### Skenario 1 — Pengguna pertama kali memesan perjalanan ulang tahun

Sarah mengunjungi untuk pertama kalinya. Agen harus menyimpan preferensinya melalui alat dan menggunakannya untuk merekomendasikan hotel.


In [ ]:
travel_agent = client.as_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### Skenario 2 — Sarah kembali beberapa minggu kemudian

Sarah memulai **utasan baru** (mensimulasikan sesi baru). Memori kerja kosong, tetapi penyimpanan preferensi jangka panjang masih memiliki informasi tentangnya. Agen harus mengambilnya dan menggunakannya untuk mempersonalisasi rekomendasi.


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## Ringkasan

Dalam pelajaran ini Anda mempelajari tiga jenis memori agen dan cara mengimplementasikannya dengan Microsoft Agent Framework:

| Jenis Memori | Mekanisme MAF | Masa Hidup |
|---|---|---|
| **Kerja** | `agent.create_session()` | Percakapan tunggal |
| **Jangka Pendek** | Konteks terakumulasi dalam sebuah thread | Tugas / sesi tunggal |
| **Jangka Panjang** | Penyimpanan eksternal diakses melalui fungsi `@tool` | Antar sesi |

### Poin Penting
1. **`agent.create_session()`** menyediakan memori kerja — agen melihat riwayat percakapan lengkap dalam sebuah sesi.
2. **Sesi baru kehilangan konteks** — tanpa memori jangka panjang agen tidak dapat mengingat percakapan sebelumnya.
3. **Fungsi `@tool`** menjembatani kesenjangan — memungkinkan agen menyimpan dan mengambil informasi dari penyimpanan permanen.
4. **Personalisasi meningkat seiring waktu** — semakin banyak preferensi disimpan, semakin baik rekomendasi agen.

### Aplikasi Dunia Nyata
- **Layanan Pelanggan**: Mengingat riwayat dan preferensi pelanggan
- **Asisten Pribadi**: Mempertahankan konteks selama hari atau minggu
- **Kesehatan**: Melacak informasi dan preferensi pasien
- **E-commerce**: Belanja yang dipersonalisasi berdasarkan riwayat

### Langkah Berikutnya
- Ganti dict dalam memori dengan basis data atau penyimpanan vektor (misalnya Azure AI Search)
- Tambahkan masa kedaluwarsa memori untuk informasi yang sensitif terhadap waktu
- Bangun sistem multi-agen dengan memori bersama
- Jelajahi notebook Cognee untuk memori berbasis knowledge-graph


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Penafian**:
Dokumen ini telah diterjemahkan menggunakan layanan terjemahan AI [Co-op Translator](https://github.com/Azure/co-op-translator). Meskipun kami berupaya untuk mencapai akurasi, harap diketahui bahwa terjemahan otomatis mungkin mengandung kesalahan atau ketidakakuratan. Dokumen asli dalam bahasa aslinya harus dianggap sebagai sumber yang sah. Untuk informasi penting, disarankan menggunakan terjemahan profesional oleh manusia. Kami tidak bertanggung jawab atas kesalahpahaman atau penafsiran yang keliru yang timbul dari penggunaan terjemahan ini.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
